# prime-rl S8: LFM2.5-VL × satelliteagent_env tool-calling RL (RTX 6000 Pro offline)

Combines:
- **S5**: satelliteagent_env (toy submit/drop, 10 scenarios) + 2-prep notebook structure
- **S6/S7**: LFM2.5-VL VLM patches + bf16/sdpa trainer config

Open question for this stage: **does the Hermes-style tool_call_parser work with
LFM2.5-VL?** The LFM2 chat template is ChatML which is structurally similar to
Qwen3, but the tool-call wire format may differ. Smoke test with
`tool_choice = "required"` to force tool emission so we can see what comes out
of the model regardless of whether it would naturally call tools.

Tiny config: `max_steps=2 batch_size=4 rollouts_per_example=2 seq_len=512`.

## 1. Install (S5 recipe + S6 patches)

In [ ]:
import os, subprocess, glob, re

PREP_MAIN = "/kaggle/input/notebooks/titanic12/prime-rl-offline-prep"
PREP_ENV  = "/kaggle/input/notebooks/titanic12/satelliteagent-env-prep"
BASE_MAIN = f"{PREP_MAIN}/output" if os.path.exists(f"{PREP_MAIN}/output") else PREP_MAIN
BASE_ENV  = f"{PREP_ENV}/output"  if os.path.exists(f"{PREP_ENV}/output")  else PREP_ENV
WHEELS_MAIN = f"{BASE_MAIN}/wheels"
WHEELS_ENV  = f"{BASE_ENV}/wheels"
MODEL_DIR = f"{BASE_MAIN}/models/LFM2.5-VL-450M"

print(f"PREP_MAIN wheels: {WHEELS_MAIN}")
print(f"PREP_ENV  wheels: {WHEELS_ENV}")
subprocess.run(f"ls {WHEELS_ENV} 2>&1 | head -5", shell=True)

SKIP_PREFIXES = (
    "torch-", "torchvision-", "torchaudio-", "nvidia_",
    "numpy-", "scipy-", "scikit_learn-", "pandas-",
)

def parse_version(name):
    m = re.match(r"([a-zA-Z0-9_]+?)-(\d+(?:\.\d+)*)", os.path.basename(name))
    if not m: return None, None
    return m.group(1).lower(), tuple(int(x) for x in m.group(2).split("."))

all_wheels = sorted(glob.glob(f"{WHEELS_MAIN}/*.whl") + glob.glob(f"{WHEELS_ENV}/*.whl"))
remaining = [w for w in all_wheels if not os.path.basename(w).startswith(SKIP_PREFIXES)]
keep = {}
for w in remaining:
    pkg, ver = parse_version(w)
    if pkg is None:
        keep[os.path.basename(w)] = (None, w); continue
    if pkg in keep and keep[pkg][0] is not None:
        if ver > keep[pkg][0]:
            keep[pkg] = (ver, w)
    else:
        keep[pkg] = (ver, w)
filtered = [v[1] for v in keep.values()]
print(f"installing {len(filtered)} wheels")
subprocess.run(
    "python3.12 -m pip install --no-index --no-build-isolation --pre --no-deps "
    + " ".join(filtered),
    shell=True, check=True,
)

# PATCH 1: register lfm2_vl in prime_rl.utils.vlm.VLM_REGISTRY
patch_registry = '''
import inspect
import prime_rl.utils.vlm as v
path = inspect.getfile(v)
content = open(path).read()
if "lfm2_vl" not in content:
    needle = "    \\"qwen3_vl\\":"
    insertion = (
        "    \\"lfm2_vl\\": VLMModelInfo(vision_encoder_attr=\\"model.vision_tower\\", language_model_attr=\\"model.language_model\\"),\\n"
        "    \\"qwen3_vl\\":"
    )
    open(path, "w").write(content.replace(needle, insertion, 1))
    print(f"PATCH 1 ok: {path}")
else:
    print("PATCH 1 skip")
'''
subprocess.run(["python3.12", "-c", patch_registry], check=True)

# PATCH 2: configure_moe_ep_backend mlp guard for non-MoE language models
patch_moe_guard = '''
import inspect
import prime_rl.trainer.model as m
path = inspect.getfile(m)
content = open(path).read()
old_block = "    for transformer_block in language_model.layers:\\n        if not isinstance(transformer_block.mlp, (MoE, LatentMoE)):\\n            continue\\n        transformer_block.mlp.set_ep_comm_backend(backend)\\n        transformer_block.mlp.set_deepep_token_chunk_size(config.deepep_token_chunk_size)"
new_block = "    for transformer_block in language_model.layers:\\n        mlp = getattr(transformer_block, \\"mlp\\", None)\\n        if mlp is None or not isinstance(mlp, (MoE, LatentMoE)):\\n            continue\\n        mlp.set_ep_comm_backend(backend)\\n        mlp.set_deepep_token_chunk_size(config.deepep_token_chunk_size)"
if old_block in content:
    open(path, "w").write(content.replace(old_block, new_block))
    print(f"PATCH 2 ok: {path}")
elif "mlp = getattr(transformer_block" in content:
    print("PATCH 2 skip (already)")
else:
    print("PATCH 2 WARN: needle not found")
'''
subprocess.run(["python3.12", "-c", patch_moe_guard], check=True)

subprocess.run(
    "python3.12 -c 'from prime_rl.utils.vlm import VLM_REGISTRY; print(\"VLM_REGISTRY:\", list(VLM_REGISTRY.keys()))'",
    shell=True, check=True,
)
subprocess.run(
    "python3.12 -c 'import satelliteagent_env; print(\"satelliteagent_env OK,\", satelliteagent_env.load_environment)'",
    shell=True, check=True,
)

## 2. Write split TOMLs (trainer / inference / orchestrator)

trainer.toml: LFM2.5-VL VLM block + bf16 + sdpa (S6/S7 と同じ)
infer.toml: `tool_call_parser = "hermes"` を試す。LFM2 が Hermes 形式互換でなければ
errror になるはずなのでログから判別する。
orch.toml: `id = "satelliteagent_env"` + `tool_choice = "required"` で smoke 検証
(base モデルは自発的に tool 呼ばないので強制が必要)。

In [ ]:
import os

PREP_MAIN = "/kaggle/input/notebooks/titanic12/prime-rl-offline-prep"
BASE_MAIN = f"{PREP_MAIN}/output" if os.path.exists(f"{PREP_MAIN}/output") else PREP_MAIN
MODEL_DIR = f"{BASE_MAIN}/models/LFM2.5-VL-450M"
VISION_ATTR = "model.vision_tower"
LM_ATTR     = "model.language_model"

TRAINER_TOML = f'''max_steps = 2

[model]
name = "{MODEL_DIR}"
seq_len = 512
attn = "sdpa"
optimization_dtype = "bfloat16"
reduce_dtype = "bfloat16"

[model.vlm]
vision_encoder_attr = "{VISION_ATTR}"
language_model_attr = "{LM_ATTR}"

[optim]
lr = 3e-6
'''

INFER_TOML = f'''gpu_memory_utilization = 0.5

[model]
name = "{MODEL_DIR}"
max_model_len = 512
enforce_eager = true
tool_call_parser = "hermes"

[server]
port = 8000
'''

ORCH_TOML = f'''max_steps = 2
batch_size = 4
seq_len = 512
rollouts_per_example = 2
filters = []

[model]
name = "{MODEL_DIR}"

[train.sampling]
max_completion_tokens = 128

[train.sampling.extra_body]
tool_choice = "required"

[[train.env]]
id = "satelliteagent_env"
'''

os.makedirs("/kaggle/working/outputs", exist_ok=True)
trainer_toml = "/kaggle/working/outputs/trainer.toml"
infer_toml   = "/kaggle/working/outputs/infer.toml"
orch_toml    = "/kaggle/working/outputs/orch.toml"
with open(trainer_toml, "w") as f: f.write(TRAINER_TOML)
with open(infer_toml,   "w") as f: f.write(INFER_TOML)
with open(orch_toml,    "w") as f: f.write(ORCH_TOML)
for label, body in [("trainer.toml", TRAINER_TOML), ("infer.toml", INFER_TOML), ("orch.toml", ORCH_TOML)]:
    print(f"=== {label} ===")
    print(body)

## 3. Run 3-process RL

In [ ]:
import os, subprocess, time, urllib.request

LOG_DIR = "/kaggle/working/outputs/proc_logs"
os.makedirs(LOG_DIR, exist_ok=True)
API_KEY = "dummy"

PREP_MAIN = "/kaggle/input/notebooks/titanic12/prime-rl-offline-prep"
BASE_MAIN = f"{PREP_MAIN}/output" if os.path.exists(f"{PREP_MAIN}/output") else PREP_MAIN
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)

env = os.environ.copy()
env.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "WANDB_MODE": "disabled",
    "CUDA_VISIBLE_DEVICES": "0",
    "VLLM_API_KEY": API_KEY,
    "HF_HOME": f"{BASE_MAIN}/hf_cache",
    "HF_HUB_CACHE": f"{BASE_MAIN}/hf_cache",
    "HF_DATASETS_CACHE": "/kaggle/working/hf_cache",
})

procs = {}
def start_bg(name, cmd):
    log_path = f"{LOG_DIR}/{name}.log"
    log_f = open(log_path, "wb")
    p = subprocess.Popen(cmd, shell=True, env=env, stdout=log_f, stderr=subprocess.STDOUT)
    procs[name] = (p, log_f, log_path)
    print(f"[{name}] started pid={p.pid} log={log_path}")
    return p

def cleanup():
    for name, (p, f, _) in procs.items():
        if p.poll() is None:
            try: p.terminate(); p.wait(timeout=10)
            except: p.kill()
        f.close()

def tail(path, n=80):
    try: return subprocess.check_output(f"tail -n {n} {path}", shell=True).decode(errors="replace")
    except Exception as e: return f"<tail error: {e}>"

outcome = "FAIL"
try:
    start_bg("inference", "python3.12 -m prime_rl.entrypoints.inference @ /kaggle/working/outputs/infer.toml")
    print("\nWaiting for vLLM ...")
    ready = False; deadline = time.time() + 600
    while time.time() < deadline:
        if procs["inference"][0].poll() is not None:
            raise RuntimeError(f"inference died early. Tail:\n{tail(procs['inference'][2], 80)}")
        try:
            req = urllib.request.Request(
                "http://localhost:8000/v1/models",
                headers={"Authorization": f"Bearer {API_KEY}"},
            )
            if urllib.request.urlopen(req, timeout=5).status == 200:
                ready = True; break
        except: pass
        time.sleep(5)
    if not ready:
        raise RuntimeError(f"inference not ready in 10min. Tail:\n{tail(procs['inference'][2], 80)}")
    print("inference is up.")

    start_bg("orchestrator", "python3.12 -m prime_rl.orchestrator.orchestrator @ /kaggle/working/outputs/orch.toml")
    time.sleep(5)
    if procs["orchestrator"][0].poll() is not None:
        raise RuntimeError(f"orchestrator died early. Tail:\n{tail(procs['orchestrator'][2], 80)}")
    print("orchestrator is up.")

    print("\nStarting trainer (live via tee)...")
    print("=" * 60)
    trainer_log = f"{LOG_DIR}/trainer.log"
    t0 = time.time()
    trainer_cmd = (
        "set -o pipefail; "
        "python3.12 -m torch.distributed.run --nproc-per-node 1 "
        "-m prime_rl.trainer.rl.train @ /kaggle/working/outputs/trainer.toml "
        f"2>&1 | tee {trainer_log}"
    )
    trainer = subprocess.run(["bash", "-c", trainer_cmd], env=env)
    elapsed = time.time() - t0
    print("=" * 60)
    print(f"\ntrainer exit={trainer.returncode}  elapsed={elapsed:.1f}s")
    if trainer.returncode != 0:
        print("\n--- inference tail ---");    print(tail(procs["inference"][2], 60))
        print("\n--- orchestrator tail ---"); print(tail(procs["orchestrator"][2], 60))
        raise RuntimeError(f"trainer failed exit={trainer.returncode}")
    outcome = "PASS"
    print("\nS8 PASS")
finally:
    cleanup()
    print(f"S8 outcome: {outcome}")

## 4. Manifest + ckpt nuke

In [ ]:
import subprocess, shutil, os
os.makedirs("/kaggle/working/outputs", exist_ok=True)
manifest = "/kaggle/working/outputs/manifest.txt"
with open(manifest, "w") as f:
    f.write("# files under /kaggle/working/ at end of notebook\n")
    for root, dirs, files in os.walk("/kaggle/working"):
        for fn in files:
            p = os.path.join(root, fn)
            try: sz = os.path.getsize(p)
            except OSError: sz = -1
            f.write(f"{sz:>12d}  {p}\n")
print("=== manifest tail ===")
subprocess.run(f"tail -n 80 {manifest}", shell=True)

ckpt_dir = "/kaggle/working/outputs/checkpoints"
if os.path.isdir(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print(f"removed: {ckpt_dir}")
else:
    print(f"no ckpt dir: {ckpt_dir}")